# MobileNetV2 Face Recognition — Student Template

## What you will build
A **face recognition model** trained to create a **128-number facial passport** for each face.
Instead of guessing a specific name out of a fixed list (classification), the model maps faces to a 128-number list so that:
- Photos of the **same** person produce **very similar** numbers (close together).
- Photos of **different** people produce **very different** numbers (far apart).

## Learning objectives
After completing this notebook you will understand:
1. Why creating 'facial passports' is better for recognizing new people than simple name-guessing.
2. How the **Rubber Band Rule (Triplet Loss)** helps the model learn.
3. How to build a **Siamese Network** where three identical workers share the same brain/memory.
4. How **Transfer Learning** (freezing the backbone's parameters) slashes calculations and saves laptop training time.
5. How to evaluate model quality using a simple yearbook search (**1-Nearest-Neighbour** classifier).
6. How to export the final model for deployment.

## Sections
1. Setup Environment
2. Load & Preprocess Dataset  ← **TODO 1**
3. Triplet Sampling            ← **TODO 2**
4. Model Architecture          ← **TODO 3**
5. Triplet Loss & Training     ← **TODO 4 & 5**
6. Evaluation                  ← **TODO 6**
7. Export

> **How to use this template:** Find every cell that contains a `# TODO` comment.
> Read the surrounding explanation, then replace `# YOUR CODE HERE` with working code.

## 1. Setup Environment
Run this cell once to install any missing libraries and import everything we need.
You do **not** need to modify this cell.

In [1]:
# Install dependencies if needed (Colab pre-installs most of these)
!pip install matplotlib scikit-learn

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


## 2. Load & Preprocess Dataset

We use the **Labeled Faces in the Wild (LFW)** dataset, fetched automatically by scikit-learn.
It contains ~13,000 celebrity face photos labelled with the person's name.
We keep only people with **at least 20 photos** so every identity has enough samples for triplets.

### Preprocessing steps you must implement (TODO 1)

| Step | What | Why |
|------|------|-----|
| **1a** | Scale pixels to `[0.0, 1.0]` | Neural networks learn faster and work more stably when pixel brightness values (0-255) are converted to decimal numbers between 0.0 and 1.0. |
| **1b** | Resize to 224 × 224 | The model's image processing layers are hardwired to process images of this specific grid size. |

In [2]:
print("Fetching LFW dataset...")
lfw_people = fetch_lfw_people(min_faces_per_person=20, resize=1.0, color=True)

X = lfw_people.images          # shape: (N, H, W, 3)  — raw uint8 pixel values 0-255
y = lfw_people.target          # shape: (N,)           — integer class index per image
target_names = lfw_people.target_names
n_classes = target_names.shape[0]

print(f"Total images: {len(X)}")
print(f"Number of classes: {n_classes}")
print(f"Image shape before preprocessing: {X.shape}")

# ---------------------------------------------------------------------------
# TODO 1a: Normalise X so that pixel values are float32 in the range [0, 1].
#
#   WHY: Raw pixel values are whole numbers from 0 to 255 (representing brightness).
#        Dividing by 255.0 converts them to decimal numbers between 0.0 and 1.0,
#        which makes calculations stable and easier for the computer chip to process.
#
#   HINT: X.astype('float32') converts the datatype to decimal representation.
#         Divide the result by the maximum possible pixel value (255.0).
# ---------------------------------------------------------------------------
X = None  # YOUR CODE HERE

# ---------------------------------------------------------------------------
# TODO 1b: Resize every image to 224 × 224 pixels if not already that size.
#
#   WHY: The model's image processing layers are hardwired to process images
#        of this specific grid size (224 wide by 224 high). Passing any other
#        size will crash the model.
#
#   The `if` guard is provided — fill in the resize call inside it.
#   HINT: tf.image.resize(tensor, [height, width]) returns a tensor;
#         call .numpy() on it to get a NumPy array back.
# ---------------------------------------------------------------------------
if X.shape[1:3] != (224, 224):
    print("Resizing images to 224x224...")
    X = None  # YOUR CODE HERE

print(f"Image shape after preprocessing: {X.shape}")

# Split into training (80%) and validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}  |  Val size: {len(X_val)}")

Fetching LFW dataset...
Total images: 3023
Number of classes: 62
Image shape before preprocessing: (3023, 125, 94, 3)


AttributeError: 'NoneType' object has no attribute 'shape'

## 3. Triplet Sampling (Selecting Groups of 3 Images)

We train our model using **triplets**: three images grouped together:

```
Anchor (A)  ─── same person ───▶  Positive (P)
Anchor (A)  ─── different person ─▶  Negative (N)
```

The model is trained to push the Anchor closer to the Positive (same person) and farther from the Negative (different person).

### Your task (TODO 2)
Complete `create_triplets()` — a helper function (generator) that yields a batch of triplets.
For each item in the batch:
1. Pick a random person.  
2. Pick two different images of that person → **Anchor** and **Positive**.  
3. Pick an image of a *different* person → **Negative**.

The surrounding infrastructure is provided for you.

In [ ]:
def create_triplets(X, y, batch_size=32):
    """
    Infinite generator that yields one batch of (anchor, positive, negative)
    triplets per iteration.

    Args:
        X (np.ndarray): All images, shape (N, 224, 224, 3).
        y (np.ndarray): Integer class labels, shape (N,).
        batch_size (int): Number of triplets per batch.

    Yields:
        Tuple of (inputs_dict, dummy_labels) where inputs_dict has keys
        'anchor', 'positive', 'negative' and dummy_labels is all zeros.
    """
    while True:
        anchors   = []
        positives = []
        negatives = []

        for _ in range(batch_size):
            # --- Anchor & Positive (same person) ----------------------------

            # Pick a random class label from the unique labels in y.
            random_label = np.random.choice(np.unique(y))

            # Find all indices in y that belong to random_label.
            # np.where(condition) returns a tuple; [0] gives the array.
            label_indices = np.where(y == random_label)[0]

            # We need at least 2 images to form an anchor–positive pair.
            if len(label_indices) < 2:
                continue

            # -------------------------------------------------------------------
            # TODO 2a: Randomly pick 2 DIFFERENT indices from label_indices.
            #          These will be the Anchor index (idx_a) and the Positive
            #          index (idx_p).
            #
            #   HINT: np.random.choice(array, size, replace=False) picks `size`
            #         elements without repetition (so a photo isn't compared to itself).
            #         Unpack the result into two variables: idx_a, idx_p
            # -------------------------------------------------------------------
            idx_a, idx_p = None, None  # YOUR CODE HERE

            # --- Negative (different person) ---------------------------------

            # -------------------------------------------------------------------
            # TODO 2b: Pick a random label that is DIFFERENT from random_label.
            #
            #   HINT: y[y != random_label] filters our list of names (y) to isolate
            #         everyone except the current target person.
            # -------------------------------------------------------------------
            negative_label = None  # YOUR CODE HERE

            # Pick a random image index that belongs to negative_label.
            idx_n = np.random.choice(np.where(y == negative_label)[0])

            # -------------------------------------------------------------------
            # TODO 2c: Append the three images to their respective lists.
            #
            #   X[idx_a] is the anchor image, X[idx_p] the positive, X[idx_n]
            #   the negative. Append each to the correct list above.
            # -------------------------------------------------------------------
            # YOUR CODE HERE

        # -----------------------------------------------------------------------
        # Provided: yield the batch as a dict so it matches the Keras Input names.
        # dummy labels (zeros) are required by the Keras training API even though
        # TripletLossLayer computes the real loss internally.
        # -----------------------------------------------------------------------
        yield (
            {
                "anchor":   np.array(anchors),
                "positive": np.array(positives),
                "negative": np.array(negatives),
            },
            np.zeros((batch_size, 1))
        )


# ---------------------------------------------------------------------------
# Provided: wrap the generator in tf.data.Dataset for type-safe, prefetchable
# pipelines. output_signature tells TF the shape and dtype of every tensor.
# ---------------------------------------------------------------------------
output_signature = (
    {
        "anchor":   tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        "positive": tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
        "negative": tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32),
    },
    tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
)

train_gen = tf.data.Dataset.from_generator(
    lambda: create_triplets(X_train, y_train, batch_size=32),
    output_signature=output_signature
)

val_gen = tf.data.Dataset.from_generator(
    lambda: create_triplets(X_val, y_val, batch_size=32),
    output_signature=output_signature
)

print("Triplet generators created successfully.")

## 4. Model Architecture (Building the Model)

We use **Transfer Learning**: we start with MobileNetV2 pre-trained on 1.2 million images, freeze its weights, and add a small custom head that produces our 128-number passport.

### ⚡ Computational Cost & Freezing the Brain
- **Model Settings (Parameters)**: MobileNetV2 has **2.2 million settings (parameters)**. This smaller size is essential for laptops and edge devices with limited calculation speed and memory.
- **Freezing the Brain (`trainable = False`)**: The backbone (main visual cortex) has already learned general visual features (like shapes, lines, and textures). By freezing these layers, we avoid recalculating adjustments for those 2.2M parameters. We only train our custom projection layer (~160,000 parameters). This cuts calculations significantly, shortening training from hours to minutes!

```
Input (224×224×3)
    │
    ▼
MobileNetV2 backbone  ← frozen (visual cortex pre-trained on ImageNet)
    │
    ▼
GlobalAveragePooling2D  ← collapses spatial shapes to flat lists
    │
    ▼
Dense(128)              ← learns to shrink info to 128 numbers
    │
    ▼
L2 Normalise            ← scales passport length to exactly 1.0
```

The same `embedding_model` is applied to all three Siamese inputs (Anchor, Positive, Negative),
**sharing weights** — this is the defining feature of a Siamese Network (three workers sharing the exact same brain).

### Your task (TODO 3)
Fill in the body of `get_embedding_model()`.

In [ ]:
def get_embedding_model():
    """
    Build and return the embedding model.

    Returns:
        keras.Model: Maps (batch, 224, 224, 3) → (batch, 128) L2-normalised embeddings.
    """
    # -----------------------------------------------------------------------
    # TODO 3a: Load the MobileNetV2 backbone.
    #
    #   We load the pre-made MobileNetV2. Think of this as the main visual
    #   cortex of the model. We discard its final classification head
    #   (include_top=False) that guesses specific labels like 'dog' or 'cat',
    #   because we want to output our custom facial passport instead.
    #
    #   Fill in the three arguments:
    #     input_shape : tuple — the image dimensions (height, width, channels)
    #     include_top : bool  — do we want the classification head?
    #     weights     : str   — which pre-trained weights to load?
    # -----------------------------------------------------------------------
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=___,   # YOUR CODE HERE
        include_top=___,   # YOUR CODE HERE
        weights=___        # YOUR CODE HERE
    )

    # -----------------------------------------------------------------------
    # TODO 3b: Freeze the backbone.
    #
    #   WHY: We freeze it (trainable = False) so we don't adjust the visual
    #        features already learned by the backbone. This saves us from doing
    #        millions of backward-pass adjustments, saving computation time.
    #
    #   Set base_model.trainable to the correct boolean value.
    # -----------------------------------------------------------------------
    base_model.trainable = ___  # YOUR CODE HERE

    # -----------------------------------------------------------------------
    # TODO 3c: Build the embedding head.
    #
    #   Follow these steps in order:
    #   1. Define an Input layer with shape (224, 224, 3).
    #   2. Apply MobileNetV2's own preprocessing (rescales to [-1, 1]):
    #        x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs * 255.0)
    #      (We multiply by 255 because our inputs are already in [0,1].)
    #   3. Pass x through base_model (with training=False so BatchNorm
    #      layers behave correctly at inference time).
    #   4. Apply layers.GlobalAveragePooling2D() to collapse the 7×7 map.
    #   5. Apply layers.Dense(128) — no activation.
    #   6. L2-normalise with a Lambda layer:
    #        layers.Lambda(lambda v: tf.math.l2_normalize(v, axis=1),
    #                      name="embedding_norm")
    #
    #   HINT: Chain the layers like this:
    #     inputs = layers.Input((224, 224, 3))
    #     x = <preprocessing>(inputs)
    #     x = <base_model>(x, training=False)
    #     x = <pooling>(x)
    #     x = <dense>(x)
    #     outputs = <l2_norm>(x)
    # -----------------------------------------------------------------------
    inputs  = None  # YOUR CODE HERE
    x       = None  # YOUR CODE HERE  (preprocessing)
    x       = None  # YOUR CODE HERE  (backbone)
    x       = None  # YOUR CODE HERE  (GlobalAveragePooling2D)
    x       = None  # YOUR CODE HERE  (Dense 128)
    outputs = None  # YOUR CODE HERE  (L2 normalise)

    return keras.Model(inputs, outputs, name="embedding_model")


# Build the embedding model and print a summary
embedding_model = get_embedding_model()
embedding_model.summary()

# ---------------------------------------------------------------------------
# Provided: Siamese Network — three Input layers that all share the same
# embedding_model weights.
# ---------------------------------------------------------------------------
anchor_input   = layers.Input((224, 224, 3), name="anchor")
positive_input = layers.Input((224, 224, 3), name="positive")
negative_input = layers.Input((224, 224, 3), name="negative")

anchor_embedding   = embedding_model(anchor_input)
positive_embedding = embedding_model(positive_input)
negative_embedding = embedding_model(negative_input)

print("\nSiamese branches built. Each branch shares the same embedding_model weights.")

## 5. Triplet Loss & Training

### 💡 Analogy: The Rubber Band Rule
- Think of **Anchor** and **Positive** as friends. We want to pull them together.
- Think of **Anchor** and **Negative** as strangers. We want to push them apart.
- Triplet loss acts like an elastic rubber band between Anchor and Positive, and a stiff wooden peg between Anchor and Negative. It penalizes the network if they aren't separated by at least a safety **margin** ($\alpha$).

### The Triplet Loss formula

$$
\mathcal{L}(A, P, N) = \max\!\Bigl(\,\underbrace{\|f(A)-f(P)\|^2}_{\text{positive distance}}
                                  -\underbrace{\|f(A)-f(N)\|^2}_{\text{negative distance}}
                                  + \alpha,\; 0\Bigr)
$$

Where:
- $f(\cdot)$ is the embedding model.
- $\alpha$ is the **margin** — a gap we force between the positive and negative distances.
- The $\max(\cdot, 0)$ clamp ensures we only penalise "easy" triplets that already satisfy
  the margin.

In plain English: *push the anchor–positive distance below the anchor–negative distance
by at least a margin $\alpha$.*

### Your tasks
- **TODO 4** — Implement `TripletLossLayer.call()`.
- **TODO 5** — Fill in the `model.fit()` training arguments.

In [ ]:
class TripletLossLayer(layers.Layer):
    """
    Custom Keras layer that computes the triplet loss and registers it
    via self.add_loss() so Keras handles backpropagation automatically.
    """

    def __init__(self, margin=0.2, **kwargs):
        super().__init__(**kwargs)
        self.margin = margin   # α in the formula above

    def call(self, inputs):
        anchor, positive, negative = inputs

        # -------------------------------------------------------------------
        # TODO 4a: Compute the POSITIVE distance — the squared distance
        #          between the anchor embedding and the positive embedding.
        #
        #   Formula: pos_dist = sum( (anchor - positive)^2 )  per sample
        #
        #   HINT: This is like finding the distance between two points on a graph
        #         for each of the 128 numbers (find the difference, square it, and sum them up).
        #         tf.square(x)             → element-wise square
        #         tf.reduce_sum(x, axis=-1) → sum across the embedding dimension
        # -------------------------------------------------------------------
        pos_dist = None  # YOUR CODE HERE

        # -------------------------------------------------------------------
        # TODO 4b: Compute the NEGATIVE distance — the squared distance
        #          between the anchor embedding and the negative embedding.
        #
        #   Same distance formula as above but using `negative` instead of `positive`.
        # -------------------------------------------------------------------
        neg_dist = None  # YOUR CODE HERE

        # -------------------------------------------------------------------
        # TODO 4c: Compute basic_loss for each sample in the batch.
        #
        #   Formula: basic_loss = pos_dist - neg_dist + self.margin
        #
        #   This will be a tensor of shape (batch_size,) — one loss value
        #   per triplet.
        # -------------------------------------------------------------------
        basic_loss = None  # YOUR CODE HERE

        # -------------------------------------------------------------------
        # TODO 4d: Clamp at zero (ignore easy triplets) and average over the
        #          batch, then register as a Keras loss.
        #
        #   Formula: loss = mean( max(basic_loss, 0) )
        #
        #   HINT: If a triplet is already correct (loss < 0), we clamp it to 0.0
        #         so the model doesn't waste effort adjusting settings it already knows.
        #         tf.maximum(x, 0.0)    → element-wise max with 0
        #         tf.reduce_mean(x)     → mean over all elements
        #
        #   After computing `loss`, call  self.add_loss(loss)  so Keras
        #   includes it in the optimisation step.
        # -------------------------------------------------------------------
        loss = None  # YOUR CODE HERE
        # YOUR CODE HERE  (self.add_loss)

        return loss


# ---------------------------------------------------------------------------
# Provided: wire the Siamese embeddings into the loss layer and compile.
# ---------------------------------------------------------------------------
loss_layer = TripletLossLayer(margin=0.2)(
    [anchor_embedding, positive_embedding, negative_embedding]
)

trainable_siamese_model = keras.Model(
    inputs=[anchor_input, positive_input, negative_input],
    outputs=loss_layer
)

trainable_siamese_model.compile(optimizer='adam')

# ---------------------------------------------------------------------------
# TODO 5: Fill in the training arguments.
#
#   - steps_per_epoch : how many batches to draw from train_gen per epoch.
#                       Start with 20 (quick experiment).
#   - epochs          : how many full passes over steps_per_epoch batches.
#                       Start with 10.
#   - validation_data : the validation generator (val_gen).
#   - validation_steps: how many batches to draw from val_gen per epoch.
#                       Start with 5.
# ---------------------------------------------------------------------------
print("Starting training...")
history = trainable_siamese_model.fit(
    train_gen,
    steps_per_epoch=___,    # YOUR CODE HERE
    epochs=___,             # YOUR CODE HERE
    validation_data=___,    # YOUR CODE HERE
    validation_steps=___    # YOUR CODE HERE
)

## 6. Evaluation (Testing the Model)

We evaluate the quality of our face passports using a **yearbook lookup (1-Nearest-Neighbour classifier)**:

1. Generate face passports for every training image.
2. For each validation image, find the single closest passport in our training database.
3. Match the validation identity to that closest training neighbor's name.
4. Compare predictions to actual names to calculate accuracy.

This tells us how well-separated our different faces are in the 128-number space.

### Your task (TODO 6)

In [ ]:
print("\nEvaluating model performance using KNN classifier on embeddings...")

# ---------------------------------------------------------------------------
# TODO 6a: Generate facial passports (embeddings) for the TRAINING and VALIDATION sets.
#
#   Use embedding_model.predict() — NOT trainable_siamese_model.
#   Pass batch_size=32 to avoid running out of memory (OOM).
#   Store results in  train_embeddings  and  val_embeddings.
# ---------------------------------------------------------------------------
print("Generating embeddings for training set...")
train_embeddings = None  # YOUR CODE HERE

print("Generating embeddings for validation set...")
val_embeddings = None  # YOUR CODE HERE

# ---------------------------------------------------------------------------
# TODO 6b: Train a 1-NN (yearbook lookup) classifier on the training embeddings.
#
#   Create a KNeighborsClassifier with:
#     n_neighbors=1   (find the single nearest yearbook neighbor)
#     metric='euclidean'
#   Then call .fit() with train_embeddings and y_train.
# ---------------------------------------------------------------------------
knn = None  # YOUR CODE HERE  (instantiate)
# YOUR CODE HERE  (.fit)

# ---------------------------------------------------------------------------
# TODO 6c: Predict labels for the validation embeddings, then compute
#           and print the accuracy.
#
#   HINT:
#     knn.predict(val_embeddings)          → predicted labels
#     accuracy_score(y_true, y_pred)       → fraction correct
# ---------------------------------------------------------------------------
y_pred = None  # YOUR CODE HERE
acc    = None  # YOUR CODE HERE

print(f"\n{'─'*50}")
print(f"Validation Accuracy (1-NN on Embeddings): {acc:.4f}")
print(f"{'─'*50}\n")

## 7. Export to TF SavedModel

We export only the **embedding model** (not the Siamese training wrapper).
The SavedModel format can be:
- Loaded directly by `FaceRecognizer` in the attendance application.
- Converted to TF-TRT for optimised GPU inference on NVIDIA hardware.

You do **not** need to modify this cell.

In [ ]:
def export_saved_model(model, filename='face_embedding_model'):
    """Export the embedding model to TensorFlow SavedModel format."""
    tf.saved_model.save(model, filename)
    print(f"Embedding model exported to '{filename}/'")
    print("You can now load it with FaceRecognizer by setting model_path to this directory.")

export_saved_model(embedding_model)